# Entrega 3 · Anexo: ¿se puede mejorar el agrupamiento?

Proyecto Integrador — Ciencia de Datos · UTN FRM · 2026 · Grupo 5K09-05

En la Entrega 3 agrupamos a los usuarios con **K-means sobre las componentes de PCA**, por separado
para géneros y para épocas. La cátedra sugirió probar **t-SNE** para ver si el agrupamiento mejora.
Este notebook hace esa prueba con los mismos datos y el mismo modelo de la entrega, y suma dos
chequeos más: un modelo de **mezclas gaussianas** con BIC y la **estabilidad** de los grupos.

**Preguntas:** si en lugar de agrupar sobre PCA agrupamos sobre un mapa de t-SNE, ¿los grupos quedan
mejor separados? Y, más en general, ¿los dos grupos de cada bloque son reales o un artefacto?

Se corre de arriba hacia abajo. Necesita `usuarios_generos.csv` y `modelado.py` en la misma carpeta.

## Hoja de ruta

| Parte | Qué hacemos |
|---|---|
| **1** | Recuperar los dos espacios y los grupos de la Entrega 3 |
| **2** | Calcular t-SNE con tres perplejidades |
| **3** | Mirar los mapas: PCA contra t-SNE |
| **4** | Agrupar sobre t-SNE y medir bien |
| **5** | Ver qué usuarios cambian de grupo |
| **6** | Mezclas gaussianas con BIC: ¿cuántos grupos pide el modelo? |
| **7** | Estabilidad: ¿salen los mismos grupos con otros usuarios? |
| **8** | Conclusión y decisión |

In [1]:
import warnings
warnings.filterwarnings("ignore")

import altair as alt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE, trustworthiness
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, silhouette_score

from modelado import SEMILLA, entrenar_desde_csv

# los mapas tienen 3.503 puntos por perplejidad y Altair corta en 5.000 filas
alt.data_transformers.disable_max_rows()
pd.set_option("display.width", 160)

# paleta del proyecto
QUEMADO, CHARTREUSE, NAVY, ELECTRICO, GRIS = "#8b3a26", "#c4a22c", "#3e3f66", "#2b30a3", "#80818f"
PERPLEJIDADES = [15, 30, 50]

## Parte 1 — Los espacios y los grupos de la Entrega 3

`entrenar_desde_csv` reproduce exactamente el modelo de la entrega: usuarios con al menos 150
ratings, afinidad amortiguada por género, proporciones por época, escalado y PCA al 80 % de varianza
en cada bloque, y K-means con k = 2. Nos quedamos con las coordenadas de PCA de cada usuario (el
espacio donde K-means agrupa) y con su grupo.

In [2]:
modelos = entrenar_desde_csv("usuarios_generos.csv")
D = modelos.entrenamiento
et = modelos.etiquetas

espacios = {
    "Géneros": {"Z": modelos.generos["preparar"].transform(D),
                "grupo": et["grupo_genero"].to_numpy(), "nombre": et["nombre_genero"].to_numpy()},
    "Épocas": {"Z": modelos.decadas["preparar"].transform(D),
               "grupo": et["grupo_decada"].to_numpy(), "nombre": et["nombre_decada"].to_numpy()},
}
pd.DataFrame({e: {"usuarios": v["Z"].shape[0], "componentes de PCA": v["Z"].shape[1]}
              for e, v in espacios.items()}).T

,usuarios,componentes de PCA
Géneros,3503,8
Épocas,3503,3


Son 3.503 usuarios. El espacio de géneros tiene 8 componentes y el de épocas, 3. Esas son las
distancias que usa K-means, y por eso **la calidad de cualquier agrupamiento se mide en este
espacio**, no en un mapa de dos dimensiones.

## Parte 2 — t-SNE

t-SNE lleva los puntos a un plano tratando de que **cada usuario conserve cerca a sus vecinos más
parecidos**. Para lograrlo deforma el resto: las distancias grandes y el tamaño de los grupos en el
mapa no significan nada. Tiene un parámetro principal, la **perplejidad**, que es más o menos
cuántos vecinos mira cada punto. No hay un valor correcto, así que probamos tres.

Una diferencia importante con PCA: t-SNE **no tiene `transform`**. No puede ubicar a un usuario
nuevo en un mapa ya hecho; habría que recalcular todo.

In [3]:
mapas = []
for espacio, v in espacios.items():
    for perp in PERPLEJIDADES:
        E = TSNE(n_components=2, perplexity=perp, init="pca", random_state=SEMILLA).fit_transform(v["Z"])
        mapas.append(pd.DataFrame({"espacio": espacio, "perplejidad": perp,
                                   "x": E[:, 0], "y": E[:, 1], "grupo": v["nombre"]}))
mapas = pd.concat(mapas, ignore_index=True)
mapas.groupby(["espacio", "perplejidad"]).size().rename("usuarios").to_frame()

usuarios
espacio perplejidad          
Géneros 15               3503
        30               3503
        50               3503
Épocas  15               3503
        30               3503
        50               3503

## Parte 3 — Mirar los mapas

A la izquierda, los dos primeros ejes de PCA; a la derecha, el mapa de t-SNE. Los colores son los
**grupos de la Entrega 3**, los mismos en los dos paneles. Los botones cambian la perplejidad del
mapa de t-SNE; clic en la leyenda para resaltar un grupo.

In [4]:
def comparar(espacio, colores):
    v = espacios[espacio]
    pca = pd.DataFrame({"x": v["Z"][:, 0], "y": v["Z"][:, 1], "grupo": v["nombre"]})
    tsne = mapas[mapas["espacio"] == espacio]
    escala = alt.Scale(domain=sorted(pca["grupo"].unique()), range=colores)
    grupo = alt.selection_point(fields=["grupo"], bind="legend")
    perp = alt.param(name="perp", value=30, bind=alt.binding_radio(options=PERPLEJIDADES, name="Perplejidad "))

    def nube(datos, titulo, ex, ey):
        return alt.Chart(datos).mark_point(filled=False, size=14, strokeWidth=0.9).encode(
            x=alt.X("x:Q", title=ex, axis=alt.Axis(grid=False)),
            y=alt.Y("y:Q", title=ey, axis=alt.Axis(grid=False)),
            color=alt.Color("grupo:N", scale=escala, title=None, legend=alt.Legend(orient="top")),
            opacity=alt.condition(grupo, alt.value(0.7), alt.value(0.04)),
        ).properties(width=330, height=300, title=titulo)

    izq = nube(pca, "PCA: los dos primeros ejes", "Eje 1", "Eje 2")
    der = nube(tsne, "t-SNE", "t-SNE 1", "t-SNE 2").transform_filter("datum.perplejidad == perp").add_params(perp)
    return (izq | der).add_params(grupo).properties(title=alt.Title(
        f"Espacio de {espacio.lower()}", anchor="start", fontSize=16)).configure_view(stroke=None)


comparar("Géneros", [CHARTREUSE, QUEMADO])

alt.HConcatChart(...)

In [5]:
comparar("Épocas", [ELECTRICO, NAVY])

alt.HConcatChart(...)

**Qué se ve.** En géneros, t-SNE dibuja una masa continua: los dos grupos son mitades de esa masa, no
islas separadas. En épocas la frontera entre grupos se ve más nítida en los dos paneles. En ningún
caso aparecen grupos nuevos que PCA estuviera escondiendo. Y al cambiar la perplejidad el dibujo
cambia bastante, aunque los datos son los mismos: es la primera señal de que la forma del mapa
depende de la perilla.

## Parte 4 — Agrupar sobre t-SNE, y medir bien

Ahora usamos el mapa como entrada de K-means (k = 2) y comparamos con los grupos de la entrega.
Medimos cuatro cosas:

- **Coincidencia (ARI)** con los grupos actuales: 1 si son idénticos, 0 si coinciden por azar.
- **Silhouette en el espacio de PCA**: la separación real, en las distancias que usa el modelo.
- **Silhouette en el plano de t-SNE**: la separación que se ve en el dibujo.
- **Confiabilidad** (`trustworthiness`): qué tanto los vecinos cercanos en el mapa también lo son en
  el espacio real (1 = todos).

In [6]:
filas = []
for espacio, v in espacios.items():
    filas.append({"espacio": espacio, "agrupamiento": "K-means sobre PCA (Entrega 3)", "perplejidad": "—",
                  "ARI vs. Entrega 3": 1.0, "silhouette en PCA": silhouette_score(v["Z"], v["grupo"]),
                  "silhouette en t-SNE": np.nan, "confiabilidad del mapa": np.nan})
    for perp in PERPLEJIDADES:
        E = mapas[(mapas["espacio"] == espacio) & (mapas["perplejidad"] == perp)][["x", "y"]].to_numpy()
        lab = KMeans(n_clusters=2, random_state=SEMILLA, n_init=10).fit_predict(E)
        filas.append({"espacio": espacio, "agrupamiento": "K-means sobre t-SNE", "perplejidad": str(perp),
                      "ARI vs. Entrega 3": adjusted_rand_score(v["grupo"], lab),
                      "silhouette en PCA": silhouette_score(v["Z"], lab),
                      "silhouette en t-SNE": silhouette_score(E, lab),
                      "confiabilidad del mapa": trustworthiness(v["Z"], E, n_neighbors=10)})
        mapas.loc[(mapas["espacio"] == espacio) & (mapas["perplejidad"] == perp), "grupo_tsne"] = lab
resultados = pd.DataFrame(filas)
resultados.round(3)

,espacio,agrupamiento,perplejidad,ARI vs. Entrega 3,silhouette en PCA,silhouette en t-SNE,confiabilidad del mapa
0,Géneros,K-means sobre PCA (Entrega 3),—,1.000,0.175,NaN,NaN
1,Géneros,K-means sobre t-SNE,15,0.506,0.134,0.356,0.950
2,Géneros,K-means sobre t-SNE,30,0.586,0.144,0.385,0.951
3,Géneros,K-means sobre t-SNE,50,0.492,0.135,0.401,0.952
4,Épocas,K-means sobre PCA (Entrega 3),—,1.000,0.422,NaN,NaN
5,Épocas,K-means sobre t-SNE,15,0.844,0.414,0.419,0.999
6,Épocas,K-means sobre t-SNE,30,0.912,0.418,0.464,0.998
7,Épocas,K-means sobre t-SNE,50,0.926,0.420,0.492,0.998


### La trampa

Si miramos la **silhouette en el plano de t-SNE**, agrupar sobre t-SNE parece duplicar la
separación en géneros. Pero t-SNE está construido para juntar lo parecido y alejar lo distinto en el
dibujo: cualquier partición medida sobre ese dibujo sale favorecida. La pregunta correcta es si los
grupos nuevos separan mejor **a los usuarios**, y eso se mide en el espacio real.

In [7]:
largo = resultados.melt(id_vars=["espacio", "agrupamiento", "perplejidad"],
                        value_vars=["silhouette en PCA", "silhouette en t-SNE"],
                        var_name="medida en", value_name="silhouette").dropna()
largo["medida en"] = largo["medida en"].str.replace("silhouette en ", "")
largo["caso"] = np.where(largo["agrupamiento"].str.contains("PCA"), "Entrega 3",
                         "t-SNE, perplejidad " + largo["perplejidad"])
orden = ["Entrega 3"] + [f"t-SNE, perplejidad {p}" for p in PERPLEJIDADES]

# una faceta exige un solo origen de datos: las tres capas parten de un gráfico sin datos
base = alt.Chart()
regla = base.transform_filter("datum.caso == 'Entrega 3'").mark_rule(
    color=GRIS, strokeDash=[4, 3]).encode(x="silhouette:Q")
puntos = base.mark_point(filled=True, size=110).encode(
    x=alt.X("silhouette:Q", title="Silhouette (más es mejor)", scale=alt.Scale(domain=[0, 0.55])),
    y=alt.Y("caso:N", sort=orden, title=None),
    color=alt.Color("medida en:N", title="Medida en el espacio de", legend=alt.Legend(orient="top"),
                    scale=alt.Scale(domain=["PCA", "t-SNE"], range=[NAVY, QUEMADO])),
    tooltip=["espacio", "caso", "medida en", alt.Tooltip("silhouette:Q", format=".3f")])
lineas = base.mark_rule(color="#d4d4da", strokeWidth=2).encode(
    x="min(silhouette):Q", x2="max(silhouette):Q", y=alt.Y("caso:N", sort=orden))

alt.layer(lineas, regla, puntos, data=largo).properties(width=520, height=150).facet(
    row=alt.Row("espacio:N", title=None, header=alt.Header(labelFontSize=13, labelAngle=0, labelAlign="left"))
).properties(title=alt.Title(
    "Medidos en el espacio real, los grupos de t-SNE separan peor que los de la Entrega 3",
    subtitle=["La silhouette en el plano de t-SNE (en burnt orange) parece mejor, pero mide el dibujo, no a los usuarios.",
              "La línea punteada es la Entrega 3."], anchor="start", fontSize=15)
).configure_view(stroke=None)

alt.FacetChart(...)

**Qué dicen los números.**

- **Géneros:** medidos en el espacio real, los grupos de t-SNE separan **peor** que los de la
  entrega, con las tres perplejidades. Y coinciden con los actuales apenas a medias (ARI entre 0,5 y
  0,6), con valores distintos según la perplejidad: el resultado depende de un parámetro que no
  tiene un valor correcto.
- **Épocas:** t-SNE encuentra casi los mismos grupos (ARI 0,84 a 0,93) y la separación real es casi
  igual. Ahí los grupos son claros y cualquier método los encuentra.
- **Confiabilidad:** los mapas son confiables para mirar vecindarios (0,95 en géneros y casi 1 en
  épocas). Sirven para ver quién se parece a quién, no para medir distancias.

## Parte 5 — ¿Qué usuarios cambian de grupo?

Con la perplejidad 30, en géneros, cruzamos el grupo de la entrega con el grupo que da t-SNE.

In [8]:
g30 = mapas[(mapas["espacio"] == "Géneros") & (mapas["perplejidad"] == 30)]
pd.crosstab(g30["grupo"], g30["grupo_tsne"].astype(int).map(lambda k: f"t-SNE grupo {k}"),
            rownames=["Entrega 3"], colnames=[None], margins=True, margins_name="Total")

,t-SNE grupo 0,t-SNE grupo 1,Total
Entrega 3,,,
"Animación, familia y aventura",1836,368,2204
"Drama, crimen e histórico",42,1257,1299
Total,1878,1625,3503


Cambian de lado 410 de los 3.503 usuarios (uno de cada ocho), y casi todos en la misma dirección: 368
usuarios del grupo de animación, familia y aventura pasan al otro grupo, contra sólo 42 en sentido
contrario. No es que t-SNE haya descubierto otra estructura: como en géneros no hay dos islas sino
una masa continua, la frontera puede pasar por muchos lugares, y t-SNE la corre según cómo dibuja el
mapa. Los usuarios que cambian son los de la zona de la frontera, los que no tienen un gusto marcado
por ninguno de los dos lados.

## Parte 6 — Mezclas gaussianas con BIC

K-means da por hecho que hay k grupos y los encuentra aunque no existan. Un **modelo de mezclas
gaussianas** (GMM) prueba otra cosa: supone que los datos salen de varias "campanas" y estima cuántas
hacen falta para describirlos. El **BIC** compara modelos con 1, 2, 3… campanas y castiga cada una de
más: **el modelo con el BIC más bajo es el preferido**.

Si los datos fueran una sola campana, el BIC elegiría 1 y diría que no hay grupos. Si fueran dos
islas, elegiría 2.

In [9]:
filas = []
for espacio, v in espacios.items():
    for k in range(1, 7):
        gm = GaussianMixture(n_components=k, covariance_type="full", n_init=5, random_state=SEMILLA).fit(v["Z"])
        filas.append({"espacio": espacio, "componentes": k, "BIC": gm.bic(v["Z"])})
bic = pd.DataFrame(filas)
bic["BIC respecto del mínimo"] = bic["BIC"] - bic.groupby("espacio")["BIC"].transform("min")
bic.pivot(index="componentes", columns="espacio", values="BIC respecto del mínimo").round(0)

espacio,Géneros,Épocas
componentes,,
1,3639.0,8698.0
2,271.0,3545.0
3,0.0,1441.0
4,239.0,634.0
5,436.0,296.0
6,698.0,0.0


In [10]:
def curva_bic(espacio, color):
    datos = bic[bic["espacio"] == espacio]
    x = alt.X("componentes:O", title="Componentes del modelo (campanas)", axis=alt.Axis(labelAngle=0))
    y = alt.Y("BIC respecto del mínimo:Q", title="BIC − el mejor (menos es mejor)")
    linea = alt.Chart(datos).mark_line(color=color, point=alt.OverlayMarkDef(color=color, size=60)).encode(
        x=x, y=y, tooltip=["componentes", alt.Tooltip("BIC:Q", format=",.0f")])
    mejor = alt.Chart(datos[datos["BIC respecto del mínimo"] == 0]).mark_point(
        size=260, color=color, strokeWidth=2).encode(x=x, y=y)
    return (linea + mejor).properties(width=300, height=220, title=espacio)


(curva_bic("Géneros", QUEMADO) | curva_bic("Épocas", ELECTRICO)).properties(title=alt.Title(
    "El BIC nunca elige una sola campana, pero tampoco se detiene en dos",
    subtitle="El círculo grande marca el modelo preferido en cada espacio.", anchor="start", fontSize=15)
).configure_view(stroke=None)

alt.HConcatChart(...)

El BIC descarta que haya **una sola** campana (el salto de 1 a 2 es enorme en los dos espacios), pero
sigue pidiendo más: 3 en géneros y 6 o más en épocas. ¿Significa que hay 3 o 6 grupos? No
necesariamente: si los datos no tienen forma de campana, el GMM agrega campanas **para copiar la
forma**, no porque haya grupos. Veamos qué separa el GMM cuando le pedimos 2 en géneros.

In [11]:
Zg = espacios["Géneros"]["Z"]
gmm2 = GaussianMixture(n_components=2, covariance_type="full", n_init=5, random_state=SEMILLA).fit(Zg)
comp = gmm2.predict(Zg)
radio = np.linalg.norm(Zg, axis=1)   # distancia al centro del espacio de géneros
resumen_gmm = pd.DataFrame({"componente": comp, "radio": radio}).groupby("componente").agg(
    usuarios=("radio", "size"), distancia_media_al_centro=("radio", "mean")).round(2)
print(f"Coincidencia del GMM de 2 componentes con la Entrega 3: ARI = "
      f"{adjusted_rand_score(espacios['Géneros']['grupo'], comp):.3f}")
resumen_gmm

Coincidencia del GMM de 2 componentes con la Entrega 3: ARI = 0.097


,usuarios,distancia_media_al_centro
componente,,
0,870,5.42
1,2633,2.58


In [12]:
vista = pd.DataFrame({"x": Zg[:, 0], "y": Zg[:, 1],
                      "componente": np.where(radio[comp == 0].mean() > radio[comp == 1].mean(),
                                             np.where(comp == 0, "Periferia (gustos marcados)", "Núcleo"),
                                             np.where(comp == 1, "Periferia (gustos marcados)", "Núcleo"))})
alt.Chart(vista).mark_point(filled=False, size=14, strokeWidth=0.9, opacity=0.6).encode(
    x=alt.X("x:Q", title="Eje 1 de géneros", axis=alt.Axis(grid=False)),
    y=alt.Y("y:Q", title="Eje 2 de géneros", axis=alt.Axis(grid=False)),
    color=alt.Color("componente:N", title=None, legend=alt.Legend(orient="top"),
                    scale=alt.Scale(domain=["Núcleo", "Periferia (gustos marcados)"], range=[NAVY, CHARTREUSE])),
).properties(width=420, height=320, title=alt.Title(
    "Con dos campanas, el GMM separa un núcleo de una periferia, no dos gustos",
    subtitle="Los colores no parten la nube en izquierda y derecha: rodean al centro.", anchor="start", fontSize=15)
).configure_view(stroke=None)

alt.Chart(...)

Con dos campanas, el GMM de géneros **no** encuentra los dos gustos de la Entrega 3 (ARI cercano a 0):
separa un **núcleo** denso de usuarios con gustos parejos de una **periferia** de usuarios con gustos
muy marcados, que quedan lejos del centro hacia cualquier lado. Es lo que pasa cuando los datos son
una masa con colas largas en lugar de islas: el BIC pide campanas para describir esas colas.

Conclusión de esta parte: el GMM **confirma que no hay islas en géneros** (lo mismo que vimos con
t-SNE) y no ofrece una partición mejor; su elección de 3 o 6 campanas describe la forma de los datos,
no grupos de usuarios.

## Parte 7 — Estabilidad: ¿salen los mismos grupos con otros usuarios?

La prueba más directa de si un agrupamiento es real: **repetirlo con otros datos** y ver si sale lo
mismo. Tomamos 30 veces el 80 % de los usuarios al azar, entrenamos K-means sólo con ellos, asignamos
a **todos** los usuarios y medimos cuánto coincide con el agrupamiento hecho con todos (ARI). Lo
hacemos para k = 2, 3, 4 y 5.

- ARI cerca de 1: salen los mismos grupos sin importar qué usuarios se usen. Los grupos son estables.
- ARI bajo o muy variable: la frontera depende de quién quedó en la muestra.

In [13]:
rng = np.random.default_rng(SEMILLA)
filas = []
for espacio, v in espacios.items():
    Z = v["Z"]
    for k in [2, 3, 4, 5]:
        completo = KMeans(n_clusters=k, random_state=SEMILLA, n_init=10).fit_predict(Z)
        for repeticion in range(30):
            idx = rng.choice(len(Z), size=int(0.8 * len(Z)), replace=False)
            parcial = KMeans(n_clusters=k, random_state=int(rng.integers(1_000_000)), n_init=10).fit(Z[idx]).predict(Z)
            filas.append({"espacio": espacio, "k": k, "repeticion": repeticion,
                          "ARI": adjusted_rand_score(completo, parcial)})
estabilidad = pd.DataFrame(filas)
estabilidad.groupby(["espacio", "k"])["ARI"].agg(["mean", "min", "max"]).round(3)

mean    min    max
espacio k                     
Géneros 2  0.962  0.929  0.998
        3  0.909  0.828  0.978
        4  0.793  0.629  0.939
        5  0.627  0.414  0.957
Épocas  2  0.986  0.963  1.000
        3  0.949  0.446  0.994
        4  0.977  0.945  0.996
        5  0.958  0.892  0.986

In [14]:
def cajas_estabilidad(espacio, color):
    datos = estabilidad[estabilidad["espacio"] == espacio]
    x = alt.X("k:O", title="Cantidad de grupos (k)", axis=alt.Axis(labelAngle=0))
    cajas = alt.Chart(datos).mark_boxplot(size=30, color=color, median={"color": "#ffffff"},
                                          outliers={"size": 18}).encode(
        x=x, y=alt.Y("ARI:Q", title="Coincidencia con el modelo completo (ARI)", scale=alt.Scale(domain=[0.3, 1])))
    marca = alt.Chart(datos[datos["k"] == 2]).mark_text(text="k elegido", dy=-14, fontSize=11, color=color).encode(
        x=x, y="max(ARI):Q")
    return (cajas + marca).properties(width=300, height=240, title=espacio)


(cajas_estabilidad("Géneros", QUEMADO) | cajas_estabilidad("Épocas", ELECTRICO)).properties(title=alt.Title(
    "Con k = 2 los grupos salen casi idénticos aunque cambien los usuarios",
    subtitle="Cada caja resume 30 repeticiones con el 80 % de los usuarios elegidos al azar.",
    anchor="start", fontSize=15)).configure_view(stroke=None)

alt.HConcatChart(...)

**Qué dicen las cajas.**

- **Géneros:** k = 2 es el más estable (ARI medio 0,96) y la estabilidad cae a medida que se piden más
  grupos (0,91 con 3, 0,79 con 4, 0,63 con 5). Aunque en géneros no haya islas, **la frontera de dos
  grupos es firme**: con otros usuarios se traza casi en el mismo lugar. Más grupos serían cortes
  arbitrarios.
- **Épocas:** k = 2 es casi perfecto (0,99). k = 4 también es estable, pero k = 3 tiene repeticiones
  que se desarman (mínimo 0,45). La elección de 2 queda respaldada; 4 sería una alternativa a explorar,
  no una mejora clara.

## Parte 8 — Conclusión

| Pregunta | Respuesta |
|---|---|
| ¿Agrupar sobre t-SNE mejora los grupos? | **No.** Medidos en el espacio real separan peor en géneros e igual en épocas. |
| ¿Por qué parece que mejora? | Porque la silhouette calculada sobre el mapa de t-SNE mide el dibujo, que t-SNE arma para que se vea separado. |
| ¿Sirve t-SNE para el recomendador? | No: no puede ubicar a un usuario nuevo sin recalcular todo el mapa. |
| ¿Qué dice el GMM con BIC? | Que los datos no son una sola campana, pero tampoco dos islas: pide más campanas para copiar la forma (núcleo y colas), no para separar gustos. |
| ¿Los grupos de la Entrega 3 son estables? | **Sí.** Con k = 2 salen casi idénticos con otros usuarios (ARI 0,96 en géneros y 0,99 en épocas), y en géneros k = 2 es el más estable de todos. |
| ¿Para qué sirve t-SNE entonces? | Para **mirar** la estructura: los mapas conservan bien a los vecinos (confiabilidad 0,95 o más). |

**Decisión:** mantenemos K-means sobre PCA con k = 2 en cada bloque, como en la Entrega 3. Ninguna de
las tres pruebas encontró un agrupamiento mejor, y la estabilidad respalda la elección. La separación
modesta en géneros (silhouette 0,175) es una propiedad de los datos —los gustos por género forman un
continuo, como mostraron t-SNE y el GMM—, y aun así la división en dos es firme: no depende de qué
usuarios se usen.